# Stage 3 — ML Modeling

**Goal:** Train two models, track every experiment with MLflow,  
pick the best one, and explain its predictions with SHAP.

| Model | Why we use it |
|---|---|
| **Logistic Regression** | Baseline — fast, interpretable, good for linearly separable features |
| **XGBoost (Gradient Boosting)** | Challenger — handles non-linear patterns, typically stronger on tabular data |

We track both with **MLflow** — every run is logged so we can compare them side by side.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
import mlflow
import mlflow.sklearn
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve
)
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

PROCESSED = Path('../data/processed')
print('Libraries loaded')

## 1. Load Processed Data

In [ ]:
X_train = np.load(PROCESSED / 'X_train.npy')
X_test  = np.load(PROCESSED / 'X_test.npy')
y_train = np.load(PROCESSED / 'y_train.npy')
y_test  = np.load(PROCESSED / 'y_test.npy')

feature_names = pd.read_csv(PROCESSED / 'feature_names.csv')['0'].tolist()

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train churn rate: {y_train.mean():.2%} (balanced by SMOTE)')
print(f'Test churn rate : {y_test.mean():.2%} (original distribution)')

## 2. Set Up MLflow

MLflow logs every experiment run — parameters, metrics, and the trained model.  
After running both models, open the UI with:  `mlflow ui`  (in terminal)

In a real company this would point to a shared tracking server, not localhost.

In [ ]:
mlflow.set_tracking_uri('../mlruns')
mlflow.set_experiment('telecom-churn-prediction')
print('MLflow tracking ready — experiment: telecom-churn-prediction')

## 3. Helper: Evaluation Plots

In [ ]:
def evaluate(model, X_test, y_test, model_name):
    """Print metrics and plot ROC + confusion matrix."""
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba)
    fpr, tpr, _ = roc_curve(y_test, y_proba)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ROC Curve
    axes[0].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'AUC = {auc:.3f}')
    axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.5)')
    axes[0].set_title(f'{model_name} — ROC Curve', fontweight='bold')
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].legend()

    # Confusion Matrix
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=['Retained', 'Churned'],
        cmap='Blues', ax=axes[1]
    )
    axes[1].set_title(f'{model_name} — Confusion Matrix', fontweight='bold')

    plt.tight_layout()
    plt.show()

    print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))
    return auc, y_proba

## 4. Model 1 — Logistic Regression (Baseline)

Logistic Regression is a great baseline:
- Fast to train
- Coefficients are directly interpretable
- Works well when features are linearly related to churn probability

We use `class_weight='balanced'` as a second layer of imbalance handling.

In [ ]:
with mlflow.start_run(run_name='logistic_regression'):

    lr = LogisticRegression(max_iter=1000, solver='saga', class_weight='balanced', random_state=42)
    lr.fit(X_train, y_train)

    auc_lr, proba_lr = evaluate(lr, X_test, y_test, 'Logistic Regression')

    # Log to MLflow
    mlflow.log_params({'solver': 'saga', 'max_iter': 1000, 'class_weight': 'balanced'})
    mlflow.log_metric('roc_auc', auc_lr)
    mlflow.sklearn.log_model(lr, 'logistic_regression')

    print(f'\nLogistic Regression — ROC-AUC: {auc_lr:.4f}')
    print('Run logged to MLflow')

## 5. Model 2 — XGBoost / Gradient Boosting (Challenger)

XGBoost builds an ensemble of decision trees where each tree corrects  
the mistakes of the previous one (boosting).

Typically outperforms Logistic Regression on tabular data because it captures  
non-linear patterns and feature interactions automatically.

In [ ]:
with mlflow.start_run(run_name='xgboost'):

    xgb_params = {
        'n_estimators':   300,
        'max_depth':      5,
        'learning_rate':  0.05,
        'subsample':      0.8,
        'colsample_bytree': 0.8,
        'use_label_encoder': False,
        'eval_metric':    'logloss',
        'random_state':   42,
        'n_jobs':         -1,
    }

    xgb = XGBClassifier(**xgb_params)
    xgb.fit(X_train, y_train, verbose=False)

    auc_xgb, proba_xgb = evaluate(xgb, X_test, y_test, 'XGBoost')

    # Log to MLflow
    mlflow.log_params(xgb_params)
    mlflow.log_metric('roc_auc', auc_xgb)
    mlflow.sklearn.log_model(xgb, 'xgboost')

    print(f'\nXGBoost — ROC-AUC: {auc_xgb:.4f}')
    print('Run logged to MLflow')

## 6. Model Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for model, proba, name, color in [
    (lr,  proba_lr,  'Logistic Regression', '#3498db'),
    (xgb, proba_xgb, 'XGBoost',             '#e74c3c'),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name}  (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random baseline')
ax.set_title('ROC Curve Comparison', fontweight='bold', fontsize=13)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'Logistic Regression AUC : {auc_lr:.4f}')
print(f'XGBoost AUC             : {auc_xgb:.4f}')
best = 'XGBoost' if auc_xgb >= auc_lr else 'Logistic Regression'
print(f'\nBest model: {best}')

## 7. SHAP — Why Does the Model Flag a Customer?

**SHAP** (SHapley Additive exPlanations) tells us how much each feature  
contributed to the model's prediction for each individual customer.

This is what you'd show a business stakeholder when they ask:  
*"Why is this customer flagged as high risk?"*

In [ ]:
# Use the best model for SHAP analysis
best_model = xgb if auc_xgb >= auc_lr else lr

explainer   = shap.TreeExplainer(best_model) if isinstance(best_model, XGBClassifier) \
              else shap.LinearExplainer(best_model, X_train)
shap_values = explainer.shap_values(X_test)

# Global feature importance — what matters most overall
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, feature_names=feature_names,
                  max_display=15, show=False)
plt.title('SHAP Feature Importance — Global', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Explain a single high-risk customer
y_proba = proba_xgb if auc_xgb >= auc_lr else proba_lr
high_risk_idx = np.argmax(y_proba)  # customer with highest predicted churn probability

print(f'Explaining customer index {high_risk_idx}')
print(f'Predicted churn probability: {y_proba[high_risk_idx]:.2%}')
print(f'Actual churn: {"Yes" if y_test[high_risk_idx] == 1 else "No"}')

shap.initjs()
shap.force_plot(
    explainer.expected_value if isinstance(explainer.expected_value, float)
    else explainer.expected_value[1],
    shap_values[high_risk_idx] if shap_values.ndim == 2
    else shap_values[1][high_risk_idx],
    X_test[high_risk_idx],
    feature_names=feature_names,
    matplotlib=True,
    show=False,
)
plt.tight_layout()
plt.show()

## 8. Save the Best Model

In [ ]:
model_path = PROCESSED / 'best_model.joblib'
joblib.dump(best_model, model_path)
np.save(PROCESSED / 'shap_values.npy', shap_values)
pd.Series(y_proba).to_csv(PROCESSED / 'churn_probabilities.csv', index=False)

print(f'Best model ({best}) saved to {model_path}')
print('Churn probabilities saved for dashboard')
print(f'\nFinal ROC-AUC: {max(auc_lr, auc_xgb):.4f}')